# 05 - Training

Training of the convolutional-recurrent classifier (`wqu.models.cred`) on one representation, with the parameters of
the `training` and `model` sections of `configs/default.yaml`:
- input standardized per frequency with mean and standard deviation of the training records;
- weighted BCE loss, bad-class weight `n_good / n_bad` of the training records;
- Adam, learning rate reduced on plateau of the validation loss, early stopping, best model kept;
- per-epoch history (loss, AUC, balanced accuracy, ECE, learning rate) and its figures;
- MC dropout predictions on the validation set.

Every run writes to its own folder `training.runs_dir/<representation>_<date>_<time>/`: copy of the configuration,
normalization statistics, `history.csv` (updated at every epoch), `checkpoint.pt` (written at every epoch),
`best_model.pt`, figures and validation predictions.

**Resuming**: with `training.resume: true`, running the notebook again after an interruption continues the most recent
unfinished run of the representation from its last completed epoch (same result as an uninterrupted run); a run is
unfinished until its validation predictions are written. The model and training settings must not have changed.

The training metrics are computed on the outputs produced while training (dropout active, weights changing within the
epoch), the validation metrics on a deterministic pass at the end of the epoch: with strong dropout the training loss can
stay above the validation loss without any anomaly.

## 0. Setup

In [1]:
from pathlib import Path

import pandas as pd

from wqu.config import load_config
from wqu.data.loading import CV_COMPONENTS
from wqu.data.splitting import SPLIT_COLUMN, TRAIN, VALIDATION, attach_split, load_split
from wqu.models.cred import build_model
from wqu.models.loss import bad_class_weight, weighted_bce_loss
from wqu.models.uncertainty import prediction_table
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    check_feature_placement,
    feature_file_path,
    read_feature_records,
)
from wqu.preprocessing.time_masks import drop_records_without_data, read_time_masks, time_mask_path
from wqu.training.data import FeatureDataset, FrequencyStatistics, frequency_statistics, make_loader
from wqu.training.metrics import classification_metrics
from wqu.training.trainer import VALIDATION_PREDICTIONS_FILE, fit, prepare_run_dir, select_device, set_seed
from wqu.visualization import apply_plot_style, plot_training_curve, plot_training_history, save_figure
import matplotlib.pyplot as plt

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
columns = config.columns
training = config.training
apply_plot_style()
set_seed(training.random_seed)
device = select_device(training.device)

features_path = feature_file_path(config.preprocessing, training.feature_source)
check_feature_placement(features_path, config.preprocessing)
records = read_feature_records(features_path, columns.text)
records[SPLIT_COLUMN] = attach_split(records, load_split(config.split.output_file, columns), columns)
print(f"Feature file: {features_path} ({len(records):,} records, {config.preprocessing.window_placement} windows); "
      f"representation: {training.representation}; device: {device}")

# Time masking: records without any recorded data in their window cannot be classified by the masked network; they
# are excluded from training and evaluation (classified bad by rule) and listed in the run folder.
time_masks = read_time_masks(time_mask_path(features_path)) if config.model.time_masking else None
excluded_records = records.iloc[:0]
if time_masks is not None:
    records, excluded_records = drop_records_without_data(records, time_masks)
print(f"Time masking: {'on' if time_masks is not None else 'off'}; records without data excluded: "
      f"{len(excluded_records):,}")

/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature file: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/processed/seisbench/full/features.h5 (100,249 records, full windows); representation: scalogram; device: mps
Time masking: on; records without data excluded: 157


## 1. Data

Per-frequency statistics from the training records only (read back from the run folder when resuming), bad-class
weight from the training labels.

In [2]:
train_records = records[records[SPLIT_COLUMN] == TRAIN]
validation_records = records[records[SPLIT_COLUMN] == VALIDATION]
bad_weight = bad_class_weight(train_records[columns.label].to_numpy())

run_dir, resumed = prepare_run_dir(training.runs_dir, training.representation, CONFIG_PATH, training.resume)
excluded_records[[columns.label, SPLIT_COLUMN]].to_csv(run_dir / "excluded_records.csv")
normalization_path = run_dir / "normalization.npz"
if normalization_path.is_file():
    statistics = FrequencyStatistics.load(normalization_path)
else:
    statistics = frequency_statistics(features_path, training.representation,
                                      train_records[FEATURE_INDEX_COLUMN].to_numpy())
    statistics.save(normalization_path)


def loader_for(subset: pd.DataFrame, shuffle: bool):
    """Loader of standardized images for a subset of the records."""
    dataset = FeatureDataset(features_path, training.representation, subset[FEATURE_INDEX_COLUMN].to_numpy(),
                             subset[columns.label].to_numpy(), statistics, time_masks)
    return make_loader(dataset, training.batch_size, shuffle, training.num_workers, training.random_seed)


train_loader = loader_for(train_records, shuffle=True)
validation_loader = loader_for(validation_records, shuffle=False)
summary = records.groupby(SPLIT_COLUMN)[columns.label].agg(records="size", bad_rate="mean")
print(f"Run folder: {run_dir} ({'resumed' if resumed else 'new run'})")
print(f"Bad-class weight: {bad_weight:.3f}")
summary.round(3)

Run folder: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/runs/scalogram_20261123_171156 (new run)
Bad-class weight: 1.790


,records,bad_rate
split,,
train,80068,0.358
validation,20024,0.359


## 2. Training

The progress is printed at every epoch; `history.csv` in the run folder is rewritten at every epoch, so the run can be
followed from outside the notebook.

In [ ]:
model = build_model(config.model, len(CV_COMPONENTS), len(statistics.mean))
result = fit(model, train_loader, validation_loader, weighted_bce_loss(bad_weight), training, bad_weight, run_dir, device)
history = result.history
print(f"Best epoch: {result.best_epoch} of {len(history)}; stopped early: {result.stopped_early}; "
      f"total time {history['seconds'].sum() / 60:.1f} min")
history.round(4)

epoch   1  train loss 0.5641  validation loss 0.5050  validation AUC 0.9034  lr 1.0e-03  154 s  (best)
epoch   2  train loss 0.5256  validation loss 0.4958  validation AUC 0.9081  lr 1.0e-03  158 s  (best)
epoch   3  train loss 0.5203  validation loss 0.5052  validation AUC 0.9034  lr 1.0e-03  156 s
epoch   4  train loss 0.5211  validation loss 0.5341  validation AUC 0.8963  lr 1.0e-03  155 s
epoch   5  train loss 0.5158  validation loss 0.5100  validation AUC 0.9059  lr 1.0e-03  157 s
epoch   6  train loss 0.5121  validation loss 0.5125  validation AUC 0.9033  lr 1.0e-03  161 s
epoch   7  train loss 0.4979  validation loss 0.4854  validation AUC 0.9129  lr 5.0e-04  165 s  (best)
epoch   8  train loss 0.4918  validation loss 0.4788  validation AUC 0.9152  lr 5.0e-04  164 s  (best)
epoch   9  train loss 0.4854  validation loss 0.4758  validation AUC 0.9154  lr 5.0e-04  169 s  (best)
epoch  10  train loss 0.4828  validation loss 0.4649  validation AUC 0.9188  lr 5.0e-04  163 s  (best)
ep

## 3. Training curves

Overview (loss and learning rate on top; AUC, balanced accuracy and ECE below) and the loss alone. The dashed line marks
the epoch of the saved model.

In [ ]:
figures_dir = run_dir / "figures"
overview = plot_training_history(history, result.best_epoch, f"Training history: {run_dir.name}")
save_figure(overview, figures_dir / "training_history.png")
save_figure(overview, figures_dir / "training_history.pdf")

loss_figure, loss_ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
plot_training_curve(history, "loss", result.best_epoch, loss_ax)
save_figure(loss_figure, figures_dir / "training_loss.png")
save_figure(loss_figure, figures_dir / "training_loss.pdf")

## 4. Validation predictions with MC dropout

For every validation record: probability of the bad class from the deterministic pass (dropout off) and its entropy;
from `model.mc_samples` stochastic passes, mean and standard deviation of the probability, predictive entropy and its
split into aleatoric part and mutual information (epistemic part); both probabilities also corrected to the class
frequencies of the training set. The table is the input of notebook 06 (evaluation).

In [ ]:
predictions = prediction_table(model, validation_loader, config.model.mc_samples, device, bad_weight)
predictions.insert(0, "record", validation_records.index.to_numpy())
predictions.insert(1, columns.label, validation_records[columns.label].to_numpy())
predictions.to_csv(run_dir / VALIDATION_PREDICTIONS_FILE, index=False)

labels = predictions[columns.label].to_numpy()
comparison = {}
for name, probability in (("deterministic", "probability_deterministic"), ("mc_dropout", "probability_mean")):
    metrics = classification_metrics(predictions[probability].to_numpy(), labels, training.calibration_bins)
    metrics["ece"] = classification_metrics(predictions[f"{probability}_corrected"].to_numpy(), labels,
                                            training.calibration_bins)["ece"]
    comparison[name] = metrics
comparison = pd.DataFrame(comparison)
comparison.to_csv(run_dir / "validation_metrics.csv")
comparison.round(4)

In [ ]:
predictions.groupby(columns.label)[["probability_mean", "probability_std", "predictive_entropy",
                                     "mutual_information"]].describe().T.round(4)